# ResNet50 image-only baseline (frozen + logistic regression)

Kvasir-VQA x1: extract ResNet50 (ImageNet) features per image, train a linear classifier on top-K answers.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "05_resnet50_image_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 128
NUM_WORKERS = 8
TOP_K = None  # use all answers
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True



Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/05_resnet50_image_only/out
Device: cuda


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)

# Resolve image paths relative to dataset root if needed
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Use all answers (no top-K filtering)
unique_answers = meta["answer_norm"].nunique()
print("Unique answers:", unique_answers)
train_k = train_df
val_k = val_df
test_k = test_df
print({"train_k": len(train_k), "val_k": len(val_k), "test_k": len(test_k)})



{'train_k': 46598, 'val_k': 5884, 'test_k': 5893, 'top_k': 200}


In [5]:
# Image embedding extraction
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    imgs = [b[1] for b in batch]
    pixel_values = torch.stack([preprocess(img) for img in imgs])
    return ids, pixel_values



In [6]:
weights = ResNet50_Weights.IMAGENET1K_V2
resnet = models.resnet50(weights=weights)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(DEVICE)
resnet.eval()

preprocess = weights.transforms()

# Compute or load cached embeddings
EMB_PATH = OUT_DIR / "image_embeddings_resnet50.npz"

# Unique images from filtered dataset
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    emb_map = {k: v for k, v in data.items()}
    print("Loaded embeddings:", len(emb_map))
else:
    ds = ImageDS(unique_imgs)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ResNet50 embeds"):
            pixels = pixels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type=="cuda"):
                feats = resnet(pixels)
            feats = feats.detach().cpu().numpy()
            for i, f in zip(ids, feats):
                emb_map[i] = f
    np.savez_compressed(EMB_PATH, **emb_map)
    print("Saved embeddings to", EMB_PATH)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/aristotle/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 38.7MB/s]


ResNet50 embeds:   0%|          | 0/204 [00:00<?, ?it/s]

Saved embeddings to /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/05_resnet50_image_only/out/image_embeddings_resnet50.npz


In [7]:
# Build feature matrices

def build_X(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_train = build_X(train_k)
X_val = build_X(val_k) if len(val_k) else None
X_test = build_X(test_k) if len(test_k) else None

y_train = train_k["answer_norm"].values
y_val = val_k["answer_norm"].values if len(val_k) else None
y_test = test_k["answer_norm"].values if len(test_k) else None

print("Shapes:", {"X_train": X_train.shape, "X_val": getattr(X_val, 'shape', None), "X_test": getattr(X_test, 'shape', None)})


Shapes: {'X_train': (46598, 2048), 'X_val': (5884, 2048), 'X_test': (5893, 2048)}


In [8]:
# Train classifier
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)


def eval_split(X, y_true, split_name):
    y_pred = clf.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    pred_df.to_csv(OUT_DIR / f"pred_{split_name}.csv", index=False)
    with open(OUT_DIR / f"metrics_{split_name}.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)
    print(split_name, metrics)


eval_split(X_train, y_train, "train")
if X_val is not None:
    eval_split(X_val, y_val, "val")
if X_test is not None:
    eval_split(X_test, y_test, "test")


/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


train {'accuracy': 0.2911498347568565, 'macro_f1': 0.013248439818847539}
val {'accuracy': 0.23130523453433038, 'macro_f1': 0.013269777566840965}
test {'accuracy': 0.23604276259969456, 'macro_f1': 0.01182237656141542}
